# Hybrid Structured + Unstructured QA System

## 1. Project Overview

**Task:** Build a single QA system that can answer questions by combining evidence from **CSV/tabular data** (structured) and **text documents** (unstructured) — routing queries to the right source, or merging both when needed.

**Why this matters:** Real-world knowledge lives in two forms:
- **Structured** — database tables, CSV files, spreadsheets (numbers, categories, relationships)
- **Unstructured** — documents, reports, wikis, emails (prose, explanations, context)

Most RAG systems handle only one. But many questions span both: *"Which product had the highest revenue last quarter, and what does the strategy report say about its growth plan?"*

**Stack:**
- `ChatOllama` + `qwen3.5:9b` — LLM for query routing, SQL generation, and answer synthesis
- `HuggingFaceEmbeddings` (`all-MiniLM-L6-v2`) — embedding model for text retrieval
- `ChromaDB` — vector store for unstructured text
- `pandas` + `sqlite3` — structured data querying

> **No API keys required.** Everything runs locally.

## 2. Learning Goals

| # | Skill |
|---|-------|
| 1 | Understand **how structured and unstructured retrieval differ** fundamentally |
| 2 | Build a **text-to-SQL generator** for structured tabular queries |
| 3 | Build a **vector retrieval pipeline** for unstructured text queries |
| 4 | Implement a **query router** that decides which source to query |
| 5 | Build a **fusion pipeline** that merges evidence from both sources |
| 6 | Compare answers from structured-only, text-only, and hybrid approaches |

## 3. Structured vs Unstructured Retrieval

### Fundamental Differences

| Dimension | Structured (Tables/CSV) | Unstructured (Text) |
|-----------|------------------------|---------------------|
| **Data format** | Rows, columns, typed values | Free-form prose |
| **Query language** | SQL / pandas operations | Semantic similarity search |
| **Strengths** | Exact numbers, aggregations, filtering | Context, explanations, reasoning |
| **Weaknesses** | No context / "why" | Imprecise for numbers, counts |
| **Match type** | Exact (WHERE x = 5) | Approximate (cosine similarity) |
| **Aggregation** | Native (SUM, AVG, COUNT) | Impossible without extraction |
| **Example query** | "Total revenue in Q3" | "Why did revenue decline?" |

### Why You Need Both

```
QUESTION: "Which product had the highest revenue and what's the growth strategy?"

STRUCTURED ANSWER:         UNSTRUCTURED ANSWER:           HYBRID ANSWER:
┌──────────────────┐       ┌──────────────────────┐       ┌──────────────────────────┐
│ Product: CloudPro │       │ "The growth strategy  │       │ CloudPro had $4.2M in    │
│ Revenue: $4.2M    │       │  outlined in the Q3   │       │ revenue (highest). The   │
│ Quarter: Q3       │       │  report focuses on    │       │ Q3 strategy report notes │
│                   │       │  enterprise expansion │       │ enterprise expansion and │
│ (no strategy info)│       │  and API partnerships"│       │ API partnerships as the  │
│                   │       │ (no exact revenue)    │       │ primary growth levers.   │
└──────────────────┘       └──────────────────────┘       └──────────────────────────┘
```

## 4. Pipeline Architecture

```
  Question
     │
     ▼
  ┌──────────────────┐
  │  QUERY ROUTER    │  LLM classifies: structured / unstructured / both
  └───┬──────┬───┬───┘
      │      │   │
  structured │  both
      │      │   │
      ▼      │   ▼
  ┌────────┐ │ ┌────────────────┐
  │ SQL    │ │ │ Both pipelines │
  │ Query  │ │ │ in parallel    │
  │ Engine │ │ └────────────────┘
  └───┬────┘ │
      │   unstructured
      │      │
      │      ▼
      │  ┌────────┐
      │  │ Vector │
      │  │ Search │
      │  └───┬────┘
      │      │
      ▼      ▼
  ┌──────────────────┐
  │  ANSWER SYNTH    │  Merge structured + unstructured evidence
  │  (LLM)           │  Produce unified answer
  └──────────────────┘
```

## 5. Environment Setup

In [ ]:
# Uncomment if any package is missing
# !pip install -q langchain langchain-ollama langchain-core langchain-community \
#     langchain-huggingface chromadb sentence-transformers pandas

print("Dependencies: langchain, chromadb, sentence-transformers, pandas, sqlite3")
print("LLM: Ollama with qwen3.5:9b (must be running locally)")

## 6. Imports

## 7. Configuration

In [ ]:
import os
import re
import json
import time
import sqlite3
import textwrap
import warnings
from collections import Counter

import pandas as pd

os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_huggingface import HuggingFaceEmbeddings
import chromadb

LLM_MODEL = "qwen3.5:9b"
EMBED_MODEL = "all-MiniLM-L6-v2"
TOP_K = 3
TEMP_SQL = 0.0
TEMP_ANSWER = 0.2

print("All imports OK")
print(f"LLM: {LLM_MODEL}")
print(f"Embeddings: {EMBED_MODEL}")
print(f"Structured backend: SQLite (in-memory)")
print(f"Unstructured backend: ChromaDB (in-memory)")

## 8. Helper Functions

In [ ]:
def clean(text: str) -> str:
    if "<think>" in text:
        text = text.split("</think>")[-1].strip()
    return text.strip()


def parse_json(text: str):
    text = clean(text)
    if "```" in text:
        text = re.sub(r"```(?:json)?\s*", "", text)
        text = text.replace("```", "")
    start = text.find("{")
    alt = text.find("[")
    if alt >= 0 and (start < 0 or alt < start):
        start = alt
    end = max(text.rfind("}"), text.rfind("]")) + 1
    if start >= 0 and end > start:
        try:
            return json.loads(text[start:end])
        except json.JSONDecodeError:
            pass
    return None


def ask(prompt: str, system: str = "", temperature: float = 0.1) -> str:
    llm = ChatOllama(model=LLM_MODEL, temperature=temperature)
    messages = []
    if system:
        messages.append(SystemMessage(content=system))
    messages.append(HumanMessage(content=prompt))
    response = llm.invoke(messages)
    return clean(response.content)


def wrap_print(text: str, width: int = 96):
    for line in text.split("\n"):
        if line.strip():
            print(textwrap.fill(line, width=width))
        else:
            print()


# Quick LLM check
test = ask("Say 'ready'.")
print(f"LLM ready: {test[:30]}")

---

# Part A — Data Sources

## 9. Structured Data — Product & Sales Tables

We simulate a company's internal database with three related tables: products, quarterly sales, and customer satisfaction scores.

In [ ]:
# --- Products table ---
products_data = {
    "product_id": ["P001", "P002", "P003", "P004", "P005"],
    "product_name": ["CloudPro", "DataSync", "SecureVault", "AnalyticsHub", "DevPipeline"],
    "category": ["Infrastructure", "Data", "Security", "Analytics", "DevTools"],
    "launch_year": [2020, 2021, 2019, 2022, 2023],
    "monthly_price_usd": [299, 149, 199, 399, 99],
    "team_size": [12, 8, 15, 10, 6],
}

# --- Quarterly sales ---
sales_data = {
    "product_id": ["P001"]*4 + ["P002"]*4 + ["P003"]*4 + ["P004"]*4 + ["P005"]*4,
    "quarter": ["Q1","Q2","Q3","Q4"]*5,
    "year": [2024]*20,
    "revenue_usd": [
        850000, 920000, 1050000, 1100000,   # CloudPro
        420000, 380000, 410000, 450000,      # DataSync
        610000, 590000, 640000, 680000,      # SecureVault
        280000, 350000, 420000, 510000,      # AnalyticsHub
        95000, 120000, 180000, 240000,       # DevPipeline
    ],
    "new_customers": [
        45, 52, 58, 61,
        30, 25, 28, 32,
        38, 35, 40, 42,
        15, 22, 30, 38,
        10, 15, 25, 35,
    ],
    "churn_rate_pct": [
        2.1, 1.8, 1.5, 1.3,
        3.5, 4.0, 3.8, 3.2,
        1.2, 1.0, 0.9, 0.8,
        2.8, 2.3, 1.9, 1.5,
        1.5, 1.2, 1.0, 0.8,
    ],
}

# --- Customer satisfaction ---
satisfaction_data = {
    "product_id": ["P001", "P002", "P003", "P004", "P005"],
    "nps_score": [72, 45, 81, 68, 77],
    "avg_support_ticket_hours": [4.2, 12.8, 2.1, 6.5, 3.8],
    "feature_requests_open": [23, 47, 12, 31, 18],
}

df_products = pd.DataFrame(products_data)
df_sales = pd.DataFrame(sales_data)
df_satisfaction = pd.DataFrame(satisfaction_data)

print("STRUCTURED DATA (3 tables)")
print(f"  products:      {len(df_products)} rows × {len(df_products.columns)} cols")
print(f"  sales:         {len(df_sales)} rows × {len(df_sales.columns)} cols")
print(f"  satisfaction:  {len(df_satisfaction)} rows × {len(df_satisfaction.columns)} cols")
print(f"\nproducts:")
print(df_products.to_string(index=False))

In [ ]:
# Load into SQLite for SQL-based querying
db = sqlite3.connect(":memory:")
df_products.to_sql("products", db, index=False, if_exists="replace")
df_sales.to_sql("sales", db, index=False, if_exists="replace")
df_satisfaction.to_sql("satisfaction", db, index=False, if_exists="replace")


def run_sql(query: str) -> pd.DataFrame:
    """Execute a SQL query and return results as DataFrame."""
    try:
        return pd.read_sql_query(query, db)
    except Exception as e:
        return pd.DataFrame({"error": [str(e)]})


# Test
test_df = run_sql("SELECT product_name, SUM(revenue_usd) as total FROM sales JOIN products USING(product_id) GROUP BY product_name ORDER BY total DESC")
print("Test SQL query — total revenue by product:")
print(test_df.to_string(index=False))

## 10. Unstructured Data — Company Documents

Reports, strategy memos, and team updates that provide context, explanations, and qualitative insights the tables don't capture.

In [ ]:
DOCUMENTS = [
    {"id": "D01", "type": "strategy", "title": "CloudPro Growth Strategy Q3 2024",
     "text": "CloudPro's growth strategy for Q3 focuses on enterprise expansion and API partnerships. The team is targeting Fortune 500 companies with a new enterprise tier offering dedicated infrastructure and SLA guarantees. An API marketplace is planned for Q4 to allow third-party integrations. The competitive moat is our multi-region failover capability which competitors lack. Pricing will increase 15% for new enterprise contracts."},

    {"id": "D02", "type": "strategy", "title": "DataSync Product Roadmap",
     "text": "DataSync is pivoting from batch-only data synchronization to real-time streaming. Customer feedback consistently highlights latency as the top pain point — current sync cycles take 15-30 minutes while competitors offer near real-time. The engineering team is rebuilding the core pipeline using Apache Kafka. Expected delivery: Q1 2025. The sales team reports losing 3 enterprise deals specifically due to latency concerns."},

    {"id": "D03", "type": "strategy", "title": "SecureVault Market Position",
     "text": "SecureVault leads the security segment with the highest NPS score across our portfolio. The zero-knowledge encryption architecture is a key differentiator — we cannot access customer data even if compelled. Compliance certifications include SOC2 Type II, HIPAA, and FedRAMP Moderate. The product is seeing strong traction in healthcare and government sectors. The team is working on post-quantum cryptography readiness."},

    {"id": "D04", "type": "strategy", "title": "AnalyticsHub Competitive Analysis",
     "text": "AnalyticsHub competes against Tableau, Looker, and PowerBI. Our advantage is embedded AI-driven insights that automatically surface anomalies without manual dashboard building. However, we lack collaborative features — Tableau and Looker offer superior multi-user editing and commenting. The Q4 roadmap prioritizes adding real-time collaboration, Slack integration, and scheduled report delivery to close this gap."},

    {"id": "D05", "type": "strategy", "title": "DevPipeline Launch Review",
     "text": "DevPipeline launched in January 2023 and has exceeded growth expectations, reaching 635 customers by Q4 2024. As our newest product, it benefits from modern architecture and developer-first design. The freemium model drives top-of-funnel growth — 40% of paid customers started on the free tier. Key risk: the CI/CD market is highly competitive with GitHub Actions and GitLab CI offering similar capabilities at lower price points."},

    {"id": "D06", "type": "report", "title": "Q3 2024 Board Report — Revenue",
     "text": "Q3 2024 company revenue reached $2.7M, a 12% increase over Q2. CloudPro remains the revenue leader driven by enterprise deal closures. AnalyticsHub showed the strongest quarter-over-quarter growth at 20%, validating the AI-first positioning. DataSync revenue declined slightly due to customer hesitancy ahead of the streaming platform migration. Management expects DataSync revenue to recover once the real-time capability launches."},

    {"id": "D07", "type": "report", "title": "Q3 2024 Board Report — Customer Health",
     "text": "Customer retention improved across all products in Q3. SecureVault achieved sub-1% churn for the first time, attributed to long-term government contracts and high switching costs. DataSync churn remains the highest at 3.8% but is trending down from 4.0% in Q2. The customer success team implemented proactive health scoring for at-risk accounts. Support ticket resolution time improved 18% company-wide."},

    {"id": "D08", "type": "report", "title": "Engineering Capacity Planning",
     "text": "Current engineering headcount is 51 across 5 product teams. CloudPro has the second-largest team (12) reflecting its revenue importance. SecureVault has the largest team (15) due to compliance and security audit requirements. DevPipeline operates lean (6 engineers) but has requested 3 additional hires for the plugin ecosystem. Cross-team initiatives include a shared authentication service and a unified billing platform."},

    {"id": "D09", "type": "memo", "title": "DataSync Customer Churn Post-Mortem",
     "text": "Analysis of Q2 DataSync churns: 60% cited sync latency as the primary reason. 25% moved to Fivetran which offers real-time CDC (change data capture). 15% consolidated to a competitor offering a broader data platform. Exit interviews reveal that customers value DataSync's reliability and support quality, but latency is a dealbreaker for real-time analytics use cases. The streaming rebuild should address the core issue."},

    {"id": "D10", "type": "memo", "title": "Enterprise Pricing Strategy Update",
     "text": "Analysis of win rates by deal size: deals under $50K/year have 65% win rate; $50K-200K have 42% win rate; above $200K have only 28% win rate. Large enterprises demand dedicated infrastructure, custom SLAs, and on-premise deployment options. CloudPro's new enterprise tier addresses this with dedicated nodes and 99.99% SLA. SecureVault's FedRAMP certification opens government deals above $500K."},

    {"id": "D11", "type": "memo", "title": "Product Portfolio Risk Assessment",
     "text": "High risk: DataSync — declining market position, latency gap vs competitors, highest churn rate. Medium risk: AnalyticsHub — strong growth but competitive pressure from well-funded incumbents (Tableau/Salesforce, Looker/Google). Low risk: CloudPro — dominant market position, growing enterprise pipeline. Low risk: SecureVault — regulatory moat, sticky contracts. Emerging risk: DevPipeline — free-tier competitors may limit pricing power long-term."},

    {"id": "D12", "type": "report", "title": "Customer Acquisition Cost Analysis",
     "text": "Customer acquisition costs (CAC) vary significantly: CloudPro CAC is $8,200 with 14-month payback. DataSync CAC is $3,800 with 9-month payback but offset by higher churn. SecureVault CAC is $12,500 with 18-month payback — justified by lowest churn and highest LTV. AnalyticsHub CAC is $5,100 with 11-month payback. DevPipeline has the lowest CAC at $950 thanks to the freemium model, with only 4-month payback."},
]

print(f"UNSTRUCTURED DATA: {len(DOCUMENTS)} documents")
print(f"Types: {dict(Counter(d['type'] for d in DOCUMENTS))}")
print(f"Avg length: {sum(len(d['text'].split()) for d in DOCUMENTS) // len(DOCUMENTS)} words")

## 11. Build the Text Retriever

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL)

chroma_client = chromadb.Client()
try:
    chroma_client.delete_collection("docs")
except Exception:
    pass

doc_collection = chroma_client.create_collection(
    name="docs", metadata={"hnsw:space": "cosine"},
)

texts = [d["text"] for d in DOCUMENTS]
metas = [{"doc_id": d["id"], "type": d["type"], "title": d["title"]} for d in DOCUMENTS]
ids = [d["id"] for d in DOCUMENTS]
vecs = embeddings.embed_documents(texts)
doc_collection.add(documents=texts, embeddings=vecs, metadatas=metas, ids=ids)

print(f"Indexed {doc_collection.count()} documents in ChromaDB")


def retrieve_text(query: str, top_k: int = TOP_K) -> list[dict]:
    """Retrieve unstructured documents by semantic similarity."""
    qv = embeddings.embed_query(query)
    results = doc_collection.query(query_embeddings=[qv], n_results=top_k)
    hits = []
    for i in range(len(results["documents"][0])):
        hits.append({
            "doc_id": results["metadatas"][0][i]["doc_id"],
            "title": results["metadatas"][0][i]["title"],
            "type": results["metadatas"][0][i]["type"],
            "text": results["documents"][0][i],
            "score": round(1 - results["distances"][0][i], 4),
        })
    return hits


# Test
test_hits = retrieve_text("Why is DataSync losing customers?")
print(f"\nTest retrieval:")
for h in test_hits:
    print(f"  [{h['doc_id']}] {h['title'][:45]}  score={h['score']:.3f}")

---

# Part B — Structured Query Engine

## 12. Text-to-SQL Generator

For structured queries, the LLM generates SQL from natural language. We provide the schema so the LLM knows what tables and columns exist.

In [ ]:
SCHEMA_DESC = """
Tables:

products (product_id TEXT PK, product_name TEXT, category TEXT, launch_year INT, monthly_price_usd INT, team_size INT)
  - 5 products: CloudPro, DataSync, SecureVault, AnalyticsHub, DevPipeline

sales (product_id TEXT FK, quarter TEXT, year INT, revenue_usd INT, new_customers INT, churn_rate_pct REAL)
  - Quarterly data for 2024 (Q1-Q4 for each product)

satisfaction (product_id TEXT FK, nps_score INT, avg_support_ticket_hours REAL, feature_requests_open INT)
  - One row per product

Joins: Use product_id to join across tables.
"""

SQL_SYSTEM = f"""You generate SQLite SQL queries from natural language questions.
Return ONLY the SQL query, nothing else. No markdown, no explanation.

DATABASE SCHEMA:
{SCHEMA_DESC}

Rules:
- Use standard SQLite syntax
- Always JOIN using product_id
- Use aliases for readability
- Return at most 10 rows unless asked for more"""


def generate_sql(question: str) -> str:
    """Convert natural language question to SQL."""
    resp = ask(question, system=SQL_SYSTEM, temperature=TEMP_SQL)
    # Clean up: extract just the SQL statement
    sql = resp.strip()
    if "```" in sql:
        sql = re.sub(r"```(?:sql)?\s*", "", sql)
        sql = sql.replace("```", "")
    # Take only the first statement
    sql = sql.strip().split(";")[0].strip()
    if not sql.upper().startswith("SELECT"):
        # Try to find SELECT in response
        match = re.search(r"(SELECT\s.+?)(?:;|$)", sql, re.IGNORECASE | re.DOTALL)
        if match:
            sql = match.group(1)
    return sql


def query_structured(question: str) -> dict:
    """Full structured query pipeline: question → SQL → execute → result."""
    sql = generate_sql(question)
    result_df = run_sql(sql)
    return {
        "source": "structured",
        "sql": sql,
        "result_df": result_df,
        "result_text": result_df.to_string(index=False),
        "row_count": len(result_df),
    }


# Test
test_sql = query_structured("Which product had the highest total revenue in 2024?")
print(f"SQL: {test_sql['sql']}")
print(f"Result:\n{test_sql['result_text']}")

In [ ]:
# Test more structured queries
test_queries = [
    "What is the average churn rate by product across all quarters?",
    "Which product gained the most new customers in Q4?",
    "Show total revenue per quarter for 2024.",
]

print("STRUCTURED QUERY TESTS")
print("=" * 80)

for q in test_queries:
    r = query_structured(q)
    print(f"\n  Q: {q}")
    print(f"  SQL: {r['sql'][:70]}")
    if "error" not in r["result_df"].columns:
        print(f"  Result ({r['row_count']} rows):")
        for line in r["result_text"].split("\n")[:5]:
            print(f"    {line}")
    else:
        print(f"  ERROR: {r['result_text'][:60]}")

---

# Part C — Query Router

## 13. Intent Classifier

The router decides whether a question needs:
- **Structured** retrieval (numbers, counts, rankings from tables)
- **Unstructured** retrieval (explanations, strategy, context from documents)
- **Both** (hybrid — needs data from tables AND context from documents)

In [ ]:
ROUTER_SYSTEM = """You classify questions by the type of data source needed.

STRUCTURED: Questions about specific numbers, rankings, comparisons, aggregations,
  counts, or filtering. These need SQL queries against tables.
  Examples: "What was Q3 revenue?", "Which product has lowest churn?"

UNSTRUCTURED: Questions about strategies, reasons, explanations, plans, context,
  or qualitative analysis. These need document search.
  Examples: "Why is DataSync losing customers?", "What's the growth strategy?"

BOTH: Questions that need numbers AND context/explanation.
  Examples: "Which product has highest revenue and what's its strategy?",
  "What's the churn rate for DataSync and why is it high?"
"""

ROUTER_PROMPT = """Classify this question: {question}

Return ONLY JSON:
{{"route": "structured" or "unstructured" or "both",
  "reasoning": "brief explanation"}}"""


def route_query(question: str) -> dict:
    """Classify whether a question needs structured, unstructured, or both."""
    resp = ask(
        ROUTER_PROMPT.format(question=question),
        system=ROUTER_SYSTEM,
        temperature=0.0,
    )
    result = parse_json(resp)
    if result and "route" in result:
        route = result["route"].lower()
        if route in ("structured", "unstructured", "both"):
            return result
    return {"route": "both", "reasoning": "Default to both for safety"}


# Test routing
test_routes = [
    "What is the total revenue for CloudPro in 2024?",
    "Why is DataSync churn so high?",
    "Which product has the lowest churn rate and what makes it sticky?",
    "How many new customers did each product gain in Q3?",
    "What is the risk assessment for our product portfolio?",
    "Which product has the highest revenue and what is its growth plan?",
]

print("QUERY ROUTING")
print("=" * 80)
for q in test_routes:
    r = route_query(q)
    print(f"  [{r['route']:>12}] {q[:60]}")

---

# Part D — Hybrid QA Pipeline

## 14. Individual Pipelines

In [ ]:
ANSWER_SYSTEM = """You answer questions using the provided evidence.
Cite data sources: [TABLE] for structured data, [DOC-XX] for documents.
Be specific with numbers from tables. Include context from documents.
If information is missing, say what's unavailable."""


def answer_structured_only(question: str) -> dict:
    """Answer using only structured/tabular data."""
    sq = query_structured(question)
    prompt = (
        f"QUESTION: {question}\n\n"
        f"SQL QUERY: {sq['sql']}\n\n"
        f"QUERY RESULT [TABLE]:\n{sq['result_text']}\n\n"
        "Answer based on this data. Cite [TABLE]."
    )
    answer = ask(prompt, system=ANSWER_SYSTEM, temperature=TEMP_ANSWER)
    return {"method": "structured_only", "answer": answer, "sql": sq["sql"],
            "data": sq["result_text"], "docs": []}


def answer_unstructured_only(question: str) -> dict:
    """Answer using only unstructured text documents."""
    hits = retrieve_text(question, top_k=3)
    context = "\n\n".join(
        f"[{h['doc_id']}] ({h['title']}):\n{h['text']}" for h in hits
    )
    prompt = (
        f"QUESTION: {question}\n\n"
        f"DOCUMENTS:\n{context}\n\n"
        "Answer based on these documents. Cite [DOC-XX]."
    )
    answer = ask(prompt, system=ANSWER_SYSTEM, temperature=TEMP_ANSWER)
    return {"method": "unstructured_only", "answer": answer, "sql": None,
            "data": None, "docs": [h["doc_id"] for h in hits]}


def answer_hybrid(question: str) -> dict:
    """Answer using both structured and unstructured sources."""
    sq = query_structured(question)
    hits = retrieve_text(question, top_k=3)

    doc_context = "\n\n".join(
        f"[{h['doc_id']}] ({h['title']}):\n{h['text']}" for h in hits
    )

    prompt = (
        f"QUESTION: {question}\n\n"
        f"STRUCTURED DATA:\n"
        f"SQL: {sq['sql']}\n"
        f"Result [TABLE]:\n{sq['result_text']}\n\n"
        f"DOCUMENTS:\n{doc_context}\n\n"
        "Synthesize an answer combining data from the table AND the documents.\n"
        "Use specific numbers from [TABLE] and context from [DOC-XX]."
    )
    answer = ask(prompt, system=ANSWER_SYSTEM, temperature=TEMP_ANSWER)
    return {"method": "hybrid", "answer": answer, "sql": sq["sql"],
            "data": sq["result_text"], "docs": [h["doc_id"] for h in hits]}


print("Three answer pipelines ready:")
print("  1. structured_only  — SQL query → table result → LLM answer")
print("  2. unstructured_only — vector search → documents → LLM answer")
print("  3. hybrid           — SQL + vector search → combined evidence → LLM answer")

## 15. Full Routed Pipeline

In [ ]:
def hybrid_qa(question: str, verbose: bool = True) -> dict:
    """Full pipeline: route → retrieve from appropriate sources → answer."""
    t0 = time.time()

    # Step 1: Route
    routing = route_query(question)
    route = routing["route"]

    if verbose:
        print(f"  Route: {route} — {routing.get('reasoning', '')[:50]}")

    # Step 2: Execute appropriate pipeline
    if route == "structured":
        result = answer_structured_only(question)
    elif route == "unstructured":
        result = answer_unstructured_only(question)
    else:  # "both"
        result = answer_hybrid(question)

    result["route"] = route
    result["routing_reason"] = routing.get("reasoning", "")
    result["time_s"] = time.time() - t0
    result["question"] = question

    if verbose:
        if result["sql"]:
            print(f"  SQL: {result['sql'][:65]}")
        if result["docs"]:
            print(f"  Docs: {result['docs']}")

    return result


print("Routed hybrid QA pipeline ready")

## 16. Demo — Different Question Types

In [ ]:
demo_questions = [
    "What is the total revenue for each product in 2024?",
    "Why is DataSync losing customers and what is the plan to fix it?",
    "Which product has the highest revenue and what is its growth strategy?",
]

print("ROUTED QA DEMO")
print("=" * 100)

for q in demo_questions:
    print(f"\nQ: {q}")
    print("-" * 80)
    r = hybrid_qa(q, verbose=True)
    print(f"  Answer:")
    wrap_print("    " + r["answer"][:300])
    print(f"  Time: {r['time_s']:.1f}s")

---

# Part E — Comparative Evaluation

## 17. Evaluation Questions

Questions designed to test whether the right source is used and whether hybrid answers are better than single-source answers.

In [ ]:
EVAL_QUESTIONS = [
    # Needs structured data only
    {"question": "What is the total revenue for CloudPro across all quarters of 2024?",
     "best_source": "structured", "type": "aggregation"},
    {"question": "Which product has the lowest average churn rate?",
     "best_source": "structured", "type": "ranking"},

    # Needs unstructured data only
    {"question": "What is the competitive positioning of AnalyticsHub?",
     "best_source": "unstructured", "type": "strategy"},
    {"question": "What are the main reasons customers are leaving DataSync?",
     "best_source": "unstructured", "type": "analysis"},

    # Needs both
    {"question": "Which product has the highest revenue and what is its growth plan?",
     "best_source": "both", "type": "hybrid_bridge"},
    {"question": "DataSync has high churn — what are the numbers and what's causing it?",
     "best_source": "both", "type": "hybrid_explain"},
    {"question": "Which product has the best customer satisfaction and why?",
     "best_source": "both", "type": "hybrid_bridge"},
    {"question": "Compare DevPipeline's growth numbers with its competitive risks.",
     "best_source": "both", "type": "hybrid_compare"},
    {"question": "What is SecureVault's revenue trend and what market factors explain it?",
     "best_source": "both", "type": "hybrid_explain"},
]

print(f"Evaluation: {len(EVAL_QUESTIONS)} questions")
print(f"Best source: {dict(Counter(q['best_source'] for q in EVAL_QUESTIONS))}")

## 18. Run All Three Pipelines on Every Question

In [ ]:
print("Running evaluation (LLM-intensive)...\n")

eval_results = []

for i, item in enumerate(EVAL_QUESTIONS, 1):
    q = item["question"]

    # Get routing decision
    routing = route_query(q)

    # Run all three pipelines
    r_struct = answer_structured_only(q)
    r_unstruct = answer_unstructured_only(q)
    r_hybrid = answer_hybrid(q)

    eval_results.append({
        "question": q,
        "best_source": item["best_source"],
        "type": item["type"],
        "routed_to": routing["route"],
        "route_correct": routing["route"] == item["best_source"],
        "structured": r_struct,
        "unstructured": r_unstruct,
        "hybrid": r_hybrid,
    })

    route_icon = "OK" if routing["route"] == item["best_source"] else "XX"
    print(f"  [{i}/{len(EVAL_QUESTIONS)}] [{route_icon}] "
          f"expected={item['best_source']:<13} routed={routing['route']:<13} "
          f"| {q[:40]}...")

## 19. Routing Accuracy

In [ ]:
correct = sum(1 for r in eval_results if r["route_correct"])
total = len(eval_results)

print("ROUTING ACCURACY")
print("=" * 60)
print(f"  Overall: {correct}/{total} ({correct/total:.0%})")
print()

# By expected source
for src in ["structured", "unstructured", "both"]:
    items = [r for r in eval_results if r["best_source"] == src]
    right = sum(1 for r in items if r["route_correct"])
    print(f"  {src:<14} {right}/{len(items)} correct")

# Show misroutes
misrouted = [r for r in eval_results if not r["route_correct"]]
if misrouted:
    print(f"\n  MISROUTED ({len(misrouted)}):")
    for r in misrouted:
        print(f"    expected={r['best_source']}, got={r['routed_to']}: {r['question'][:55]}")

## 20. LLM-as-Judge — Answer Quality by Source

In [ ]:
JUDGE_SYSTEM = """You compare answers to a question that may need both data and context.
Evaluate which answer is more complete, accurate, and informative.
Value specific numbers (from tables) AND explanatory context (from documents)."""

JUDGE_PROMPT = """QUESTION: {question}

ANSWER A ({method_a}):
{answer_a}

ANSWER B ({method_b}):
{answer_b}

Which answer better addresses the question? Return ONLY JSON:
{{"winner": "A" or "B" or "tie",
  "reasoning": "brief explanation",
  "quality_a": 1-5,
  "quality_b": 1-5}}"""


# For hybrid questions, compare hybrid vs each single source
hybrid_qs = [r for r in eval_results if r["best_source"] == "both"]

print("HYBRID vs SINGLE-SOURCE QUALITY (for questions needing both)")
print("=" * 90)

hybrid_vs_struct = {"hybrid": 0, "structured": 0, "tie": 0}
hybrid_vs_unstruct = {"hybrid": 0, "unstructured": 0, "tie": 0}

for r in hybrid_qs:
    q = r["question"]
    q_short = textwrap.shorten(q, 55, placeholder="...")

    # Hybrid vs Structured
    j1 = parse_json(ask(
        JUDGE_PROMPT.format(
            question=q,
            method_a="structured_only",
            answer_a=r["structured"]["answer"][:350],
            method_b="hybrid",
            answer_b=r["hybrid"]["answer"][:350],
        ),
        system=JUDGE_SYSTEM, temperature=0.0,
    )) or {"winner": "tie"}
    w1 = "structured" if j1.get("winner") == "A" else "hybrid" if j1.get("winner") == "B" else "tie"
    hybrid_vs_struct[w1] += 1

    # Hybrid vs Unstructured
    j2 = parse_json(ask(
        JUDGE_PROMPT.format(
            question=q,
            method_a="unstructured_only",
            answer_a=r["unstructured"]["answer"][:350],
            method_b="hybrid",
            answer_b=r["hybrid"]["answer"][:350],
        ),
        system=JUDGE_SYSTEM, temperature=0.0,
    )) or {"winner": "tie"}
    w2 = "unstructured" if j2.get("winner") == "A" else "hybrid" if j2.get("winner") == "B" else "tie"
    hybrid_vs_unstruct[w2] += 1

    print(f"  {q_short}")
    print(f"    hybrid vs struct: {w1:<12}  hybrid vs unstruct: {w2}")

print(f"\n  SUMMARY (across {len(hybrid_qs)} hybrid questions):")
print(f"  Hybrid vs Structured:   hybrid={hybrid_vs_struct['hybrid']}, struct={hybrid_vs_struct['structured']}, tie={hybrid_vs_struct['tie']}")
print(f"  Hybrid vs Unstructured: hybrid={hybrid_vs_unstruct['hybrid']}, unstruct={hybrid_vs_unstruct['unstructured']}, tie={hybrid_vs_unstruct['tie']}")

## 21. Side-by-Side Answer Comparison

In [ ]:
# Show detailed side-by-side for 2 hybrid questions
print("DETAILED ANSWER COMPARISON")
print("=" * 100)

for r in hybrid_qs[:2]:
    print(f"\nQ: {r['question']}")
    print(f"Expected source: {r['best_source']}, Routed to: {r['routed_to']}")

    print(f"\n  STRUCTURED ONLY:")
    if r["structured"]["sql"]:
        print(f"    SQL: {r['structured']['sql'][:65]}")
    wrap_print("    " + r["structured"]["answer"][:200])

    print(f"\n  UNSTRUCTURED ONLY:")
    print(f"    Docs: {r['unstructured']['docs']}")
    wrap_print("    " + r["unstructured"]["answer"][:200])

    print(f"\n  HYBRID:")
    if r["hybrid"]["sql"]:
        print(f"    SQL: {r['hybrid']['sql'][:65]}")
    print(f"    Docs: {r['hybrid']['docs']}")
    wrap_print("    " + r["hybrid"]["answer"][:250])

    print()

---

# Part F — Analysis & Learning Notes

## 22. When Each Approach Wins

| Question Pattern | Best Source | Why |
|-----------------|-----------|-----|
| "How much revenue?" | Structured | Exact number from table |
| "Which product is highest/lowest?" | Structured | Aggregation + sorting |
| "Why is churn high?" | Unstructured | Explanation, root cause |
| "What's the growth strategy?" | Unstructured | Strategic context |
| "What's the revenue AND strategy?" | Both | Number + context |
| "Is churn improving and what's driving it?" | Both | Trend data + causal analysis |

### Failure Modes

| Mode | Example | Fix |
|------|---------|-----|
| **Text can't do math** | Asking documents "total revenue" | Route to SQL |
| **Tables lack context** | Asking SQL "why did revenue grow" | Route to documents |
| **Wrong routing** | "What's our best product?" (ambiguous) | Default to hybrid |
| **SQL error** | Generated SQL has syntax bug | Retry with error feedback |
| **Conflicting sources** | Table says X, document says Y | Prioritize table for numbers, docs for reasoning |

## 23. Common Mistakes

| Mistake | Why It's Wrong | Better Approach |
|---------|---------------|----------------|
| Embedding tabular data as text | Loses column relationships and makes aggregation impossible | Keep tables in SQL, embed only text |
| Using only vector search for everything | Numbers get hallucinated; "total revenue" returns wrong values | Route numerical queries to SQL |
| Hard-coding route rules | "if contains 'revenue' → SQL" breaks on "explain revenue decline" | LLM-based routing with schema awareness |
| No error handling for bad SQL | Model generates invalid SQL, pipeline crashes | Catch SQL errors, retry or fallback |
| Showing raw SQL results to user | DataFrame dumps are not user-friendly | LLM synthesizes natural language answer |
| Not exposing the schema to the LLM | Text-to-SQL fails without knowing table names and columns | Always include schema in SQL generation prompt |

## 24. Production Improvement Ideas

1. **SQL retry with error feedback** — if SQL fails, send the error back to the LLM for correction
2. **Schema auto-detection** — read table schemas dynamically so the system works with any database
3. **Confidence-based routing** — if router is uncertain, run both sources and merge
4. **Table serialization for hybrid** — convert SQL results to sentences for richer synthesis
5. **Query history** — cache SQL results for repeated queries
6. **Multi-table join hints** — detect when a question spans multiple tables and generate JOINs
7. **Guardrails** — block DELETE/UPDATE/DROP; only allow SELECT queries

## 25. Exercises

### Exercise 1
Add **SQL retry with error handling**: when the generated SQL fails, capture the error message, send it back to the LLM with the original question and schema, and ask it to generate a corrected query. Allow up to 2 retries.

### Exercise 2
Build a **table summarizer**: automatically generate natural language descriptions of each table (column meanings, value ranges, row count) and include them in the text vector store so users can ask "what data do we have?" and get useful answers.

### Exercise 3
Implement **conflicting evidence detection**: when the hybrid pipeline retrieves both SQL results and documents, check if they present contradictory information (e.g., table shows revenue up, document says revenue declined). Flag contradictions in the answer.

### Mini Challenge
Extend the system to handle **time-series questions** like "How has CloudPro's revenue trended over the year?" — generate SQL for the data, retrieve documents about CloudPro's strategy, and produce an answer that combines the trend data with explanatory context.

In [ ]:
print("SESSION SUMMARY")
print("=" * 65)
print(f"Structured data: 3 tables ({len(df_products) + len(df_sales) + len(df_satisfaction)} total rows)")
print(f"Unstructured data: {len(DOCUMENTS)} documents")
print(f"Evaluation: {len(EVAL_QUESTIONS)} questions")
print()
print(f"Routing accuracy: {correct}/{total} ({correct/total:.0%})")
print()
print(f"Hybrid vs single-source (on {len(hybrid_qs)} questions needing both):")
print(f"  Hybrid beat structured: {hybrid_vs_struct['hybrid']}/{len(hybrid_qs)}")
print(f"  Hybrid beat unstructured: {hybrid_vs_unstruct['hybrid']}/{len(hybrid_qs)}")
print()
print(f"Components built:")
print(f"  1. SQL query engine    — text-to-SQL with schema awareness")
print(f"  2. Text retriever      — ChromaDB vector search on documents")
print(f"  3. Query router        — LLM classifies structured/unstructured/both")
print(f"  4. Hybrid synthesizer  — combines SQL results + documents into one answer")
print(f"  5. Routed pipeline     — end-to-end: route → retrieve → answer")
print(f"  6. LLM-as-judge       — quality comparison across approaches")

## 26. Key Takeaways

| # | Takeaway |
|---|----------|
| 1 | **Structured and unstructured retrieval are fundamentally different** — SQL is exact and aggregate-capable; vector search is approximate and context-rich |
| 2 | **Neither alone is sufficient** — tables give precise numbers but no explanations; documents give context but imprecise numbers |
| 3 | **Query routing is essential** — sending a "total revenue" question to vector search produces hallucinated numbers |
| 4 | **Hybrid answers are superior for complex questions** — they combine data precision with explanatory context |
| 5 | **Text-to-SQL needs the schema** — the LLM must know table names, column names, and types to generate correct SQL |
| 6 | **SQL generation is fragile** — small syntax errors break the pipeline; always add error handling and retries |
| 7 | **Don't embed tables as text** — this destroys column relationships and makes aggregation impossible |
| 8 | **Default to hybrid when routing is uncertain** — it's better to include unnecessary context than miss critical information |
| 9 | **Tables are authoritative for numbers** — when SQL results and documents conflict on specific figures, trust the table |
| 10 | **Real-world QA systems always span both data types** — building hybrid pipelines is a production-essential skill |

---

*This notebook is part of the [Machine Learning Projects](https://github.com/pypi-ahmad/Machine-Learning-Projects) repository.*